# 01 — DNRTI: A Real NER Dataset for CTI

**Goal:** download DNRTI, parse it, inspect it, and save it as a HuggingFace `DatasetDict` the later notebooks can load with one line.

## Why DNRTI?

In CTI 101 (notebook 05) we trained NER on a toy dataset with a handful of entity types. DNRTI gives us:

- **~300 real CTI reports** annotated at the token level
- **13 entity types**: `HackOrg`, `SamFile`, `Tool`, `Time`, `Idus`, `Org`, `Way`, `Exp`, `Features`, `Area`, `Purp`, `OffAct`, `SecTeam`
- **BIO-tagged**, one `token TAB tag` per line, blank line = sentence boundary

That's the same shape the CTI 101 NER pipeline already expects — so in notebook 04 we'll reuse that code almost verbatim.

## What this notebook does

1. Clone the DNRTI GitHub repo into `data/dnrti/`
2. Extract the `.rar` archive (requires system `unrar`)
3. Parse the BIO files into sentences of `(tokens, tags)`
4. Inspect class balance and sentence-length distribution
5. Save a `DatasetDict` to `processed/dnrti/` for downstream notebooks

**Runs once.** Skip to notebook 02 on subsequent sessions.

## Step 1 — Clone the DNRTI repository

The canonical repo is `SCreaMxp/DNRTI-A-Large-scale-Dataset-for-Named-Entity-Recognition-in-Threat-Intelligence`. It's small (a few MB) so we clone the whole thing.

In [1]:
import os
from pathlib import Path

DATA_DIR = Path('data/dnrti')
REPO_DIR = DATA_DIR / 'repo'
REPO_URL = 'https://github.com/SCreaMxp/DNRTI-A-Large-scale-Dataset-for-Named-Entity-Recognition-in-Threat-Intelligence.git'

DATA_DIR.mkdir(parents=True, exist_ok=True)

if REPO_DIR.exists():
    print(f'Already cloned at {REPO_DIR}')
else:
    os.system(f'git clone --depth 1 {REPO_URL} {REPO_DIR}')
    print('Clone complete.')

print('\nRepo contents:')
for p in sorted(REPO_DIR.iterdir()):
    print(f'  {p.name}')

Cloning into 'data/dnrti/repo'...


Clone complete.

Repo contents:
  .git
  DNRTI.rar
  README.md


## Step 2 — Extract the `.rar` archive

DNRTI ships as a single `.rar` file. We use the `rarfile` Python library, which shells out to the system `unrar` binary.

If this cell fails with `RarCannotExec: Unrar not installed?`, install it:

```bash
sudo apt install unrar    # Debian/Ubuntu
brew install unrar        # macOS
```

In [3]:
import rarfile

RAR_PATH = REPO_DIR / 'DNRTI.rar'
EXTRACT_DIR = DATA_DIR / 'extracted'
EXTRACT_DIR.mkdir(exist_ok=True)

with rarfile.RarFile(RAR_PATH) as rf:
    rf.extractall(EXTRACT_DIR)

print('Extracted files:')
for p in sorted(EXTRACT_DIR.rglob('*')):
    if p.is_file():
        size_kb = p.stat().st_size / 1024
        print(f'  {p.relative_to(EXTRACT_DIR)}  ({size_kb:.1f} KB)')

Extracted files:
  test.txt  (180.2 KB)
  train.txt  (1437.2 KB)
  valid.txt  (178.3 KB)


## Step 3 — Locate the train/valid/test splits

DNRTI's archive contains pre-split BIO files (typically `train.txt`, `valid.txt`, `test.txt`). File names vary between releases, so we search for them rather than hard-coding.

In [4]:
bio_files = {}
for p in EXTRACT_DIR.rglob('*.txt'):
    stem = p.stem.lower()
    for key in ('train', 'valid', 'dev', 'test'):
        if key in stem:
            # Normalize 'dev' -> 'validation'
            split = 'validation' if key in ('valid', 'dev') else key
            bio_files[split] = p
            break

print('Detected splits:')
for split, path in bio_files.items():
    print(f'  {split}: {path.relative_to(EXTRACT_DIR)}')

assert 'train' in bio_files and 'test' in bio_files, \
    'Could not locate train/test files in extracted archive.'

Detected splits:
  train: train.txt
  test: test.txt
  validation: valid.txt


## Step 4 — Parse BIO format into `(tokens, tags)` sentences

BIO format: each line is `token<TAB>tag` (or space-separated), and a blank line marks a sentence boundary.

```
APT29      B-HackOrg
used       O
Cobalt     B-Tool
Strike     I-Tool

The        O
campaign   O
targeted   O
...
```

In [5]:
def read_bio(path: Path):
    """Yield (tokens, tags) per sentence. Handles tab or whitespace separators."""
    tokens, tags = [], []
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.rstrip('\n')
            if not line.strip():
                if tokens:
                    yield tokens, tags
                    tokens, tags = [], []
                continue
            parts = line.split('\t') if '\t' in line else line.split()
            if len(parts) < 2:
                continue
            tokens.append(parts[0])
            tags.append(parts[-1])
        if tokens:
            yield tokens, tags

parsed = {split: list(read_bio(path)) for split, path in bio_files.items()}

for split, sents in parsed.items():
    n_sent = len(sents)
    n_tok = sum(len(toks) for toks, _ in sents)
    print(f'{split:12s}  {n_sent:5d} sentences  {n_tok:7d} tokens')

train          5251 sentences   140348 tokens
test            664 sentences    17716 tokens
validation      662 sentences    17602 tokens


### Peek at one sentence

Sanity check — do the tokens and tags line up, and do the tags look like BIO?

In [6]:
sample_tokens, sample_tags = parsed['train'][0]
for tok, tag in zip(sample_tokens[:30], sample_tags[:30]):
    marker = '  <--' if tag != 'O' else ''
    print(f'  {tok:20s}  {tag}{marker}')

  The                   O
  admin@338             B-HackOrg  <--
  has                   O
  largely               O
  targeted              O
  organizations         O
  involved              O
  in                    O
  financial             B-Idus  <--
  ,                     O
  economic              B-Idus  <--
  and                   O
  trade                 B-Idus  <--
  policy                I-Idus  <--
  ,                     O
  typically             O
  using                 O
  publicly              B-Tool  <--
  available             I-Tool  <--
  RATs                  I-Tool  <--
  such                  O
  as                    O
  Poison                B-Tool  <--
  Ivy                   I-Tool  <--
  ,                     O
  as                    O
  well                  O
  some                  O
  non-public            B-Tool  <--
  backdoors             I-Tool  <--


## Step 5 — Build the tag vocabulary

We collect every tag seen in training, sort for determinism, and keep `O` at index 0 so downstream models default to "no entity".

In [7]:
from collections import Counter

tag_counter = Counter()
for tokens, tags in parsed['train']:
    tag_counter.update(tags)

# O first, then B-/I- tags sorted alphabetically
tag_list = ['O'] + sorted(t for t in tag_counter if t != 'O')
tag2id = {t: i for i, t in enumerate(tag_list)}
id2tag = {i: t for t, i in tag2id.items()}

print(f'Total tag classes: {len(tag_list)}')
print('\nTag frequencies in training set:')
for tag, count in tag_counter.most_common():
    print(f'  {tag:15s}  {count:6d}')

Total tag classes: 27

Tag frequencies in training set:
  O                110435
  B-HackOrg          3419
  B-Tool             2449
  B-Area             2171
  B-OffAct           1412
  I-Tool             1386
  B-Idus             1349
  B-Time             1328
  I-Purp             1231
  B-SamFile          1221
  I-Features         1178
  B-Org              1113
  B-Exp              1068
  B-SecTeam           997
  I-HackOrg           977
  I-Org               958
  I-OffAct            851
  B-Way               828
  I-Area              816
  B-Features          812
  I-Time              794
  I-Way               785
  B-Purp              721
  I-SamFile           556
  I-Exp               538
  I-Idus              482
  I-SecTeam           473


**What to notice:**

- `O` dominates — most tokens are non-entities. This matters for evaluation: token-level accuracy is misleading; we'll use span-level F1 (notebook 05).
- Entity types are imbalanced. `HackOrg`, `Tool`, `SamFile` are usually well-represented; `Purp`, `Features` are rarer — expect lower per-class F1 for the tail.
- Every entity type should have both `B-` and `I-` variants in the counts. A missing `I-` for some type means that entity is always single-token in training.

## Step 6 — Sentence-length distribution

Relevant for notebook 04 (picking `max_length` for the tokenizer) and notebook 08 (deciding whether Longformer is even needed for NER).

In [8]:
import numpy as np

lengths = [len(toks) for toks, _ in parsed['train']]
lengths_arr = np.array(lengths)

print(f'Sentence length (train, in whitespace tokens):')
print(f'  min    {lengths_arr.min()}')
print(f'  median {int(np.median(lengths_arr))}')
print(f'  mean   {lengths_arr.mean():.1f}')
print(f'  p95    {int(np.percentile(lengths_arr, 95))}')
print(f'  p99    {int(np.percentile(lengths_arr, 99))}')
print(f'  max    {lengths_arr.max()}')
print(f'\n% over 128 whitespace tokens: {(lengths_arr > 128).mean()*100:.1f}%')
print(f'% over 256 whitespace tokens: {(lengths_arr > 256).mean()*100:.1f}%')

Sentence length (train, in whitespace tokens):
  min    2
  median 25
  mean   26.7
  p95    45
  p99    55
  max    82

% over 128 whitespace tokens: 0.0%
% over 256 whitespace tokens: 0.0%


Keep these numbers in mind when we set `max_length` in notebook 04. After subword tokenization the count roughly doubles, so a safe `max_length` is usually ~2× the p99 here.

## Step 7 — Save as a HuggingFace `DatasetDict`

We persist the parsed data so every downstream notebook can do:

```python
from datasets import load_from_disk
ds = load_from_disk('processed/dnrti')
```

…with zero reparsing.

In [9]:
import json
from datasets import Dataset, DatasetDict, Features, Sequence, Value, ClassLabel

features = Features({
    'tokens': Sequence(Value('string')),
    'ner_tags': Sequence(ClassLabel(names=tag_list)),
})

def to_dataset(sents):
    rows = {
        'tokens': [toks for toks, _ in sents],
        'ner_tags': [[tag2id[t] for t in tags] for _, tags in sents],
    }
    return Dataset.from_dict(rows, features=features)

ds = DatasetDict({split: to_dataset(sents) for split, sents in parsed.items()})

OUT_DIR = Path('processed/dnrti')
OUT_DIR.parent.mkdir(exist_ok=True)
ds.save_to_disk(str(OUT_DIR))

# Also save the tag vocabulary as plain JSON for quick lookups
with open(OUT_DIR / 'tag_vocab.json', 'w') as f:
    json.dump({'tag_list': tag_list, 'tag2id': tag2id}, f, indent=2)

print(f'Saved to {OUT_DIR}')
print(ds)

Saving the dataset (0/1 shards):   0%|          | 0/5251 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/664 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/662 [00:00<?, ? examples/s]

Saved to processed/dnrti
DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 5251
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 664
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 662
    })
})


## Step 8 — Smoke test: load it back

Confirm the saved dataset round-trips cleanly. If this cell works, notebook 04 will too.

In [10]:
from datasets import load_from_disk

reloaded = load_from_disk(str(OUT_DIR))
ex = reloaded['train'][0]
label_names = reloaded['train'].features['ner_tags'].feature.names

print('First training example (first 15 tokens):')
for tok, tag_id in list(zip(ex['tokens'], ex['ner_tags']))[:15]:
    print(f'  {tok:20s}  {label_names[tag_id]}')

First training example (first 15 tokens):
  The                   O
  admin@338             B-HackOrg
  has                   O
  largely               O
  targeted              O
  organizations         O
  involved              O
  in                    O
  financial             B-Idus
  ,                     O
  economic              B-Idus
  and                   O
  trade                 B-Idus
  policy                I-Idus
  ,                     O


## Summary

| | CTI 101 (nb 05) | CTI 201 (DNRTI) |
|---|---|---|
| Sentences | ~dozens | hundreds–thousands |
| Entity types | ~3 | 13 |
| Source | hand-crafted toy | real CTI reports |
| Class balance | roughly flat | heavily skewed |

**Next:** notebook 02 loads TRAM II and builds the MITRE tactic label space for multi-label classification.

**Reuse:** `processed/dnrti/` is consumed by notebooks 04 and 05.